# AgriVision AI — Train your crop-yield ML model

**Run every code cell from top to bottom.** Click the ▶ button on each cell. You do not need to edit the code.

This notebook downloads a public historical dataset, trains a Random Forest regression model, evaluates it, and downloads the two files required by the Streamlit website.

**Dataset limitation:** records are country-level and cover 1990–2013. This is for a college demonstration, not a guaranteed farm-level forecast.


## 1. Install libraries

In [ ]:
!pip -q install pandas numpy scikit-learn joblib


## 2. Download and inspect the dataset

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/kdeeey/cropai/main/data/yield_df.csv"
df = pd.read_csv(DATA_URL)
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]

print("Dataset size:", df.shape)
display(df.head())
print(df.columns.tolist())


## 3. Clean and prepare data

In [ ]:
required = ["Area", "Item", "Year", "average_rain_fall_mm_per_year",
            "pesticides_tonnes", "avg_temp", "hg/ha_yield"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[required].copy().dropna()
numeric = ["Year", "average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp", "hg/ha_yield"]
for col in numeric:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna()
df = df[df["hg/ha_yield"] > 0].copy()

print("Rows used:", len(df))
print("Countries:", df["Area"].nunique())
print("Crops:", df["Item"].nunique())
display(df.head())


## 4. Train the Random Forest model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

features = ["Area", "Item", "Year", "average_rain_fall_mm_per_year",
            "pesticides_tonnes", "avg_temp"]
target = "hg/ha_yield"
X = df[features]
y = df[target]

preprocess = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="ignore"), ["Area", "Item"]),
    ("numbers", "passthrough", ["Year", "average_rain_fall_mm_per_year",
                                  "pesticides_tonnes", "avg_temp"])
])

model = Pipeline([
    ("preprocess", preprocess),
    ("regressor", RandomForestRegressor(
        n_estimators=160, min_samples_leaf=2, random_state=42, n_jobs=-1
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("Model training complete!")
print(f"MAE: {mae:,.2f} hg/ha")
print(f"RMSE: {rmse:,.2f} hg/ha")
print(f"R2 score: {r2:.4f}")


## 5. Save model and website settings

In [ ]:
import json
import joblib

joblib.dump(model, "yield_model.joblib")

def stats(col):
    return {"min": float(df[col].min()),
            "max": float(df[col].max()),
            "median": float(df[col].median())}

metadata = {
    "countries": sorted(df["Area"].astype(str).unique().tolist()),
    "crops": sorted(df["Item"].astype(str).unique().tolist()),
    "year_min": int(df["Year"].min()),
    "year_max": int(df["Year"].max()),
    "numeric_ranges": {
        "average_rain_fall_mm_per_year": stats("average_rain_fall_mm_per_year"),
        "pesticides_tonnes": stats("pesticides_tonnes"),
        "avg_temp": stats("avg_temp")
    },
    "metrics": {"mae": float(mae), "rmse": float(rmse), "r2": float(r2)},
    "rows": int(len(df))
}

with open("metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Created yield_model.joblib and metadata.json")


## 6. Download the two trained model files

In [ ]:
from google.colab import files
files.download("yield_model.joblib")
files.download("metadata.json")


## 7. Next: upload files to GitHub

Create a public repository called `agrivision-ai`. Upload these four files into the repository's main/root folder:
- `app.py` (from this project ZIP)
- `requirements.txt` (from this project ZIP)
- `yield_model.joblib` (downloaded in Step 6)
- `metadata.json` (downloaded in Step 6)

Then open https://share.streamlit.io → sign in with GitHub → **Create app** → choose your repository → branch `main` → main file path `app.py` → **Deploy**.

Official deployment guide: https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app
